# 07 — The dollar–oil break: transient or structural?

Finding 5 was the project's key result: the dollar–oil correlation *flipped sign*
between the explore period (WTI vs USD ≈ −0.5) and the 2022+ holdout (≈ +0.1).
That was a descriptive split across two fixed windows. This notebook makes it
rigorous, in three escalating steps:

1. **Rolling correlation** through the whole sample — is the flip a temporary
   shock that's reverting, or a lasting break?
2. **Structural-break test** — date the break(s) precisely rather than assuming
   2022-01-01.
3. **Markov-switching model** — let the data define the regimes and estimate the
   transition dynamics, instead of splitting on a calendar date.

Uses the FULL sample here deliberately: the point is to characterize the break
across the explore/holdout boundary, so both sides are in view. No prediction is
made — this is structural characterization.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path.cwd().parent))
import config

daily = pd.read_parquet(config.DATA/"commodities_daily.parquet")
monthly = pd.read_parquet(config.DATA/"commodities_monthly.parquet")
plt.rcParams["figure.figsize"]=(12,5)

# daily returns for resolution
oil_ret = np.log(daily["wti"].where(daily["wti"]>0)).diff()
usd_ret = daily["usd_broad"].pct_change()
print("daily overlap:", pd.concat([oil_ret,usd_ret],axis=1).dropna().shape[0], "days")

## Step 1 — Rolling correlation: is the flip reverting?

126-day (~6mo) rolling correlation of WTI vs broad-USD daily returns, across the
whole sample. The explore/holdout boundary and the 2022 shock are marked. The
question: after the 2022 flip toward positive, does it revert toward the
historical negative, or stay broken?

In [ ]:
pair = pd.concat([oil_ret.rename("oil"), usd_ret.rename("usd")], axis=1).dropna()
roll = pair["oil"].rolling(126).corr(pair["usd"])

fig,ax=plt.subplots()
roll.plot(ax=ax, color="darkred", lw=0.9)
ax.axhline(0, color="k", lw=0.6)
ax.axvline(pd.Timestamp(config.HOLDOUT_START), color="steelblue", ls="--", lw=1.2, label="explore/holdout boundary")
ax.axvspan(pd.Timestamp("2022-02-01"), pd.Timestamp("2022-12-31"), alpha=0.12, color="orange", label="2022 energy shock")
ax.set_title("Rolling 6mo correlation: WTI vs broad USD (daily)")
ax.set_ylabel("correlation"); ax.legend(); plt.tight_layout(); plt.show()

# recent trajectory: mean rolling corr by year, 2020 onward
recent = roll.loc["2020":].groupby(roll.loc["2020":].index.year).mean()
print("Mean 6mo rolling WTI-USD correlation by year:")
print(recent.round(3).to_string())

**Read:** if the yearly means climb back toward negative after 2022, the flip
was a transient shock. If they hover near zero or stay positive through 2024–25,
it's a more lasting structural change (candidate explanations: sanctions-driven
oil trade shifts, dollar-oil decoupling).

## Step 2 — Structural-break detection

Where did the relationship actually break? Two approaches: a rolling-correlation
changepoint (via `ruptures` if available), and a formal test on the correlation
series itself.

In [ ]:
# changepoint on the rolling-correlation series
try:
    import ruptures as rpt
    sig = roll.dropna().values
    algo = rpt.Pelt(model="rbf").fit(sig)
    bkps = algo.predict(pen=10)
    dates = roll.dropna().index
    print("Detected changepoints in WTI-USD rolling correlation:")
    for b in bkps[:-1]:
        print("  ", dates[b].date())
except ImportError:
    print("ruptures not installed -- skip changepoint (pip install ruptures for this).")
    print("Step 3 (Markov-switching) provides the regime dating instead.")

## Step 3 — Markov-switching regression: let the data define regimes

Instead of splitting on 2022-01-01, fit a 2-regime Markov-switching model where
WTI return is regressed on USD return, allowing the *coefficient* (the beta) to
switch between regimes. This estimates: (a) the two betas, (b) when each regime
was active, (c) how persistent each is.

In [ ]:
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
import statsmodels.api as sm

# monthly for stability of the switching model
m_oil = np.log(monthly["wti"].where(monthly["wti"]>0)).diff()
m_usd = monthly["usd_broad"].diff()
d = pd.concat([m_oil.rename("oil"), m_usd.rename("usd")], axis=1).dropna()

# 2-regime switching regression: oil ~ const + usd, with switching slope & variance
X = sm.add_constant(d["usd"])
mod = MarkovRegression(d["oil"].values, k_regimes=2, exog=X.values,
                       switching_variance=True)
res = mod.fit()
print(res.summary())

In [ ]:
# smoothed regime probabilities, aligned to the model's index
sp = res.smoothed_marginal_probabilities
# statsmodels returns shape (nobs, k_regimes); take regime-1 column robustly
p_reg1 = np.asarray(sp)[:, 1] if np.asarray(sp).ndim == 2 else np.asarray(sp[1])
smoothed = pd.Series(p_reg1, index=d.index)

fig,ax=plt.subplots()
smoothed.plot(ax=ax, color="purple")
ax.axvline(pd.Timestamp(config.HOLDOUT_START), color="steelblue", ls="--", lw=1.2,
           label="explore/holdout boundary")
ax.set_title("Smoothed probability of regime 2 (WTI-USD switching model)")
ax.set_ylabel("P(regime 2)"); ax.legend(); plt.tight_layout(); plt.show()

# report the USD slope in each regime (defensive to param layout)
print("Expected regime durations (months):", np.round(res.expected_durations,1))
print("\nModel parameters (see summary above for labels):")
print(np.round(res.params, 3))
print("\nInterpretation: compare the two USD-slope params -- one regime should")
print("show the normal negative dollar-oil beta, the other the broken (>=0) beta.")
print("The purple plot dates when regime 2 was active; overlap with 2022+ =")
print("the break regime coincides with the energy shock.")

**Read:** compare the USD coefficient across the two regimes — one should be
clearly negative (normal), the other near-zero or positive (the break). The
smoothed-probability plot dates when the break regime was active. If it lights up
strongly in 2022+ and the expected duration is short, that supports 'transient
shock'; if the break regime persists through the holdout, 'structural.'

## Verdict (fill after running)

- **Rolling correlation trajectory 2022→25:** reverting to negative, or stuck?
- **Break date(s):** when did the relationship actually change?
- **Markov regimes:** the two USD-betas, and how long the break regime lasts.

**Synthesis:** this turns finding 5 from "it flipped across two windows" into a
dated, modeled structural break with estimated regime dynamics — the rigorous
version, and the basis for the cross-asset regime work in notebook 08.